# FFHQ 128x128: multiscale NFlow and pullback geometry

Uses the two multiscale transform classes unchanged from
`mnist_multi_ellipsoid_starflow.ipynb`, with the geometry experiments from
`cifar10_additive_nflow.ipynb`. No star/ellipsoid stages are imported.

## Data setup (once, outside this notebook)
Official source: https://github.com/NVlabs/ffhq-dataset
FFHQ supplies 70,000 official 128x128 PNG thumbnails (approximately 1.95 GB).
Review the official dataset terms and individual image licenses before use.
For example, in a terminal:

```sh
git clone https://github.com/NVlabs/ffhq-dataset.git
cd ffhq-dataset
python -m pip install -r requirements.txt
python download_ffhq.py --thumbs
```

Set `data_root` below to the resulting `thumbnails128x128` directory.
The notebook reads PNGs recursively (flat or nested layout), without requiring
class folders. It does not download data or resize higher-resolution originals.
Every loaded image must already be 128x128; images are converted to RGB.

## Small-start defaults
- A deterministic random subset of 5,000 available images, split 80/20.
  Set `max_images=None` to use all available images. The subset is a pilot,
  not a full-FFHQ evaluation; no official train/test split is implied.
- Training batch size 8; geometry analysis on 32 fixed held-out images.
  These are starting choices, not a GPU-memory guarantee. Geometry routines
  may be substantially more expensive than training; lower analysis size/rank
  if necessary (rank must remain smaller than analysis size).
- Kernel 3, hidden channels 64, 3 flow steps per scale, 3 scales, Adam 0.001,
  float32, 100 epochs. Change `n_flows` to 6 for the depth ablation.
  The source MNIST YAML was not attached, so these are constructor defaults.
- Scale shapes: (12,64,64), (24,32,32), (48,16,16).
- Normalization to [-1,1], no augmentation/dequantization, seed 31.
  Reported losses are continuous normalized-input NLL, not discrete-image bpd.
- The original unshuffled training order and training-NLL checkpoint selection
  are preserved; held-out NLL is additionally logged each epoch to CSV.
- Separate checkpoint folders encode architecture, learning rate, batch size
  and the selected data manifest. Resume requires an identical manifest.

## Geometry and execution
Preserves sampling, pullback geodesics/barycentre, rank-20 tangent PGA,
isometrized geodesics/PGA with five intervals, Euclidean baseline, and RMSE
comparisons. The source's shared pullback barycentre remains the base point
for all low-rank solvers, including Euclidean PCA (not recentered PCA).
Plots alone inverse-normalize and clip. Calculations use unclipped tensors.

Run in the original project environment with its `src` package, PyTorch,
torchvision, nflows, NumPy, Pillow, matplotlib and tqdm. No YAML is needed.
A small real-data forward/inverse check initializes ActNorm before training.
All outputs are cleared. Notebook structure and syntax are validated, but
training and the custom geometry implementation have not been executed here.


In [ ]:
import os
import glob
import csv
import hashlib
from pathlib import Path
from PIL import Image
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
torch.set_default_dtype(torch.float32)
from torch.utils.data import DataLoader, random_split, Dataset
import torchvision
import matplotlib.pyplot as plt
from nflows.flows import Flow
from nflows.distributions import StandardNormal
from nflows.transforms import OneByOneConvolution, SqueezeTransform
from nflows.transforms.normalization import ActNorm
from tqdm import tqdm

from src.diffeomorphisms.image.nflow import NFlowImageDiffeomorphism
from src.dimension_reduction.principal_geodesic_analysis.image.l2_tangent_space_pca import l2TangentSpacePCAImageSolver
from src.manifolds.euclidean.image.pullback.standard import StandardPullbackImageEuclidean
from src.manifolds.euclidean.image.standard import StandardImageEuclidean
from src.manifolds.isometrized_euclidean.image import l2IsometrizedImageEuclidean
from src.transforms.composite.image import ImageCompositeTransform
from src.transforms.image import ImageTransform
from src.transforms.image.parity_conv.linear_2d import Linear2DParityConvImageTransform
from src.transforms.image.parity_conv.simple_tanh_2d import SimpleTanh2DParityConvImageTransform

device = "cuda" if torch.cuda.is_available() else "cpu"

# Set random seed for reproducibility
seed = 31
torch.manual_seed(seed)
np.random.seed(seed)

# Experiment settings
size = 128
C = 3
results_folder = os.path.join("results",f"ffhq_{size}x{size}","mnist-multiscale-nflow-pullback")
train_model = True
resume_training = True

plt.rcParams['font.size'] = 20  # Change this value to your desired font size
os.makedirs(results_folder, exist_ok=True)

# Edit these paths/settings before running.
data_root = Path("data/ffhq/thumbnails128x128")
max_images = 5000  # None: all available images
batch_size = 8
analysis_batch_size = 32  # independent of training batch size
pga_rank = 20
num_workers = 0


### Learning Geometry ###

In [ ]:
# %pip install -U huggingface_hub

In [ ]:
# from pathlib import Path
# from collections import Counter
# import shutil
# import zipfile

# from huggingface_hub import hf_hub_download
# from tqdm.auto import tqdm

# ffhq_root = Path("data/ffhq").resolve()
# data_root = ffhq_root / "thumbnails128x128"
# data_root.mkdir(parents=True, exist_ok=True)

# archive = hf_hub_download(
#     repo_id="nuwandaa/ffhq128",
#     repo_type="dataset",
#     filename="thumbnails128x128.zip",
#     revision="1b277b6efe3ba86ff0c712b88753892ca5c4e7a3",
#     local_dir=str(ffhq_root),
# )

# with zipfile.ZipFile(archive) as z:
#     images = [
#         entry for entry in z.infolist()
#         if not entry.is_dir()
#         and entry.filename.lower().endswith(".png")
#         and "__MACOSX" not in entry.filename.split("/")
#     ]

#     if not images:
#         raise RuntimeError("No PNG images found in the archive.")

#     # Flatten image folders, but refuse ambiguous duplicate filenames.
#     counts = Counter(Path(entry.filename).name for entry in images)
#     duplicates = [name for name, count in counts.items() if count > 1]
#     if duplicates:
#         raise RuntimeError(
#             f"Duplicate image filenames in archive: {duplicates[:5]}"
#         )

#     for entry in tqdm(images, desc="Extracting FFHQ thumbnails"):
#         target = data_root / Path(entry.filename).name

#         # Skip files already extracted completely on a previous run.
#         if target.exists() and target.stat().st_size == entry.file_size:
#             continue

#         temporary = target.with_suffix(".png.part")
#         with z.open(entry) as source, temporary.open("wb") as destination:
#             shutil.copyfileobj(source, destination)
#         temporary.replace(target)

# print("Image directory:", data_root)
# print("PNG images found:", sum(1 for _ in data_root.glob("*.png")))

In [ ]:
H, W = size, size
# Fixed affine normalization: same convention for both RGB datasets.
image_mean = (0.5, 0.5, 0.5)
image_std = (0.5, 0.5, 0.5)
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(image_mean, image_std),
])
class FFHQThumbnails(Dataset):
    def __init__(self, root, transform, max_images=None, seed=31):
        self.root = Path(root).expanduser().resolve()
        self.transform = transform
        if not self.root.is_dir():
            raise FileNotFoundError(
                f"FFHQ thumbnails not found: {self.root}. Download the official "
                "128x128 thumbnails and set data_root; see setup instructions above.")
        files = sorted(p for p in self.root.rglob("*")
                       if p.is_file() and p.suffix.lower() == ".png")
        if not files:
            raise FileNotFoundError(f"No PNG thumbnails found under {self.root}")
        available = len(files)
        if max_images is not None:
            if max_images < 1:
                raise ValueError("max_images must be positive or None")
            chosen = torch.randperm(available, generator=torch.Generator().manual_seed(seed))
            files = [files[i] for i in sorted(chosen[:max_images].tolist())]
        self.files = files
        print(f"Using {len(files):,} of {available:,} available FFHQ thumbnails")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        path = self.files[index]
        with Image.open(path) as image:
            if image.size != (W, H):
                raise ValueError(f"Expected {(W,H)} thumbnail, got {image.size}: {path}")
            x = self.transform(image.convert("RGB"))
        return x, -1  # unlabeled; placeholder keeps the inherited training API

full_dataset = FFHQThumbnails(data_root, transform, max_images, seed)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
if train_size < 6 or val_size < analysis_batch_size:
    raise ValueError("Need >=6 training images and enough held-out images for analysis")
if not (20 < analysis_batch_size and 1 <= pga_rank < analysis_batch_size):
    raise ValueError("Use analysis_batch_size >=21 and 1 <= pga_rank < analysis_batch_size")
data_train, data_val = random_split(full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(seed))
train_loader = DataLoader(data_train, batch_size=batch_size, shuffle=False, num_workers=num_workers)
validation_loader = DataLoader(data_val, batch_size=batch_size, shuffle=False, num_workers=num_workers)
val_loader = DataLoader(data_val, batch_size=analysis_batch_size, shuffle=False, num_workers=num_workers)
train_images, _ = next(iter(train_loader))
val_images, val_labels = next(iter(val_loader))

manifest = {
    "files": [str(p.relative_to(full_dataset.root)) for p in full_dataset.files],
    "train_indices": list(data_train.indices), "val_indices": list(data_val.indices),
    "seed": seed,
}
manifest_text = __import__("json").dumps(manifest, sort_keys=True)
data_id = hashlib.sha256(manifest_text.encode()).hexdigest()[:16]
print(f"Train: {train_size:,}; validation: {val_size:,}; geometry: {len(val_images)}")

# Display only: all calculations use unclipped, normalized RGB tensors.
def display_image(image):
    image = torch.as_tensor(image).detach().cpu().reshape(C, H, W)
    mean = torch.tensor(image_mean).view(C, 1, 1)
    std = torch.tensor(image_std).view(C, 1, 1)
    return (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

for images in (train_images, val_images):
    fig, axes = plt.subplots(1, 6, figsize=(12, 3))
    for i, ax in enumerate(axes):
        ax.imshow(display_image(images[i]))
        ax.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
import torch.nn as nn

from src.transforms.image.parity_conv import ParityConvImageTransform


class TanhResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, residual_scale=0.1):
        super().__init__()

        padding = kernel_size // 2
        self.residual_scale = residual_scale

        self.net = nn.Sequential(
            nn.Tanh(),
            nn.Conv2d(
                channels,
                channels,
                kernel_size=kernel_size,
                padding=padding,
            ),
            nn.Tanh(),
            nn.Conv2d(
                channels,
                channels,
                kernel_size=kernel_size,
                padding=padding,
            ),
        )

    def forward(self, x):
        return x + self.residual_scale * self.net(x)


class ResidualTanh2DParityConvImageTransform(ParityConvImageTransform):
    def __init__(
        self,
        in_channels,
        height,
        width,
        hidden_channels=128,
        kernel_size=3,
        parity=0,
        n_blocks=3,
        residual_scale=0.1,
    ):
        if kernel_size < 1 or kernel_size % 2 != 1:
            raise ValueError("kernel_size must be a positive odd integer")
        if n_blocks < 1:
            raise ValueError("n_blocks must be >= 1")

        padding = kernel_size // 2

        output_conv = nn.Conv2d(
            hidden_channels,
            in_channels,
            kernel_size=kernel_size,
            padding=padding,
        )

        # Start the additive update at zero.
        nn.init.zeros_(output_conv.weight)
        nn.init.zeros_(output_conv.bias)

        conv_network = nn.Sequential(
            nn.Conv2d(
                in_channels,
                hidden_channels,
                kernel_size=kernel_size,
                padding=padding,
            ),
            *[
                TanhResidualBlock(
                    channels=hidden_channels,
                    kernel_size=kernel_size,
                    residual_scale=residual_scale,
                )
                for _ in range(n_blocks)
            ],
            nn.Tanh(),
            output_conv,
        )

        super().__init__(
            in_channels,
            height,
            width,
            conv_network,
            parity,
        )

class MultiLevelImageTransform(ImageTransform):
    def __init__(self, in_channels, height, width, image_transforms):
        """
        :param image_transforms: list of ImageTransform objects, one for each level of the multi-level transform.
        """
        self.L = len(image_transforms)
        assert image_transforms[0].C == in_channels * 4
        assert image_transforms[0].H == height // 2
        assert image_transforms[0].W == width // 2
        # assert that the image transform dimensions shrink by a factor of 2 at each level, but channels grow by a factor of 2 at each level
        assert all(image_transforms[i].C == image_transforms[i-1].C * 2 for i in range(1, self.L))
        assert all(image_transforms[i].H == image_transforms[i-1].H // 2 for i in range(1, self.L))
        assert all(image_transforms[i].W == image_transforms[i-1].W // 2 for i in range(1, self.L))
        super().__init__(in_channels, height, width)
        self.image_transforms = nn.ModuleList(image_transforms)

        self.squeeze = SqueezeTransform(factor=2)
        

    def forward(self, inputs, context=None):
        batch_size = inputs.shape[0]
        outputs = inputs
        total_logabsdet = inputs.new_zeros(batch_size)
        splits = []
        for i in range(self.L):
            # squeeze
            outputs, log_abs_det = self.squeeze(outputs, None)
            total_logabsdet += log_abs_det

            # flow step
            outputs, log_abs_det = self.image_transforms[i](outputs, None)
            total_logabsdet += log_abs_det

            # split
            split, outputs = torch.chunk(outputs, 2, dim=1)
            splits.append(split)

        # add context if provided
        if context is not None:
            assert context.shape[0] == batch_size
            assert context.shape[1] == self.image_transforms[-1].H
            assert context.shape[2] == self.image_transforms[-1].W
            context_ = torch.zeros_like(outputs)
            context_[:, 0] = context
            outputs = outputs - context_

        # iterate through splits to get outputs in correct order
        for i in reversed(range(self.L)):
            # concatenate split back to outputs
            outputs = torch.cat([splits[i], outputs], dim=1)
            # unsqueeze
            outputs, _ = self.squeeze.inverse(outputs, None)

        return outputs, total_logabsdet
    
    def inverse(self, inputs, context=None):
        # iterate through splits to get outputs in correct order
        outputs = inputs
        splits = []
        for i in range(self.L):
            # squeeze
            outputs, _ = self.squeeze(outputs, None)

            split, outputs = torch.chunk(outputs, 2, dim=1)
            splits.append(split)

        # remove context if provided
        if context is not None:
            assert context.shape[0] == inputs.shape[0]
            assert context.shape[1] == self.image_transforms[-1].H
            assert context.shape[2] == self.image_transforms[-1].W
            context_ = torch.zeros_like(outputs)
            context_[:, 0] = context
            outputs = outputs + context_

        total_logabsdet = inputs.new_zeros(inputs.shape[0])
        for i in reversed(range(self.L)):
            # concatenate
            outputs = torch.cat([splits[i], outputs], dim=1)

            # flow step
            outputs, log_abs_det = self.image_transforms[i].inverse(outputs, None)
            total_logabsdet += log_abs_det

            # unsqueeze
            outputs, log_abs_det = self.squeeze.inverse(outputs, None)
            total_logabsdet += log_abs_det

        return outputs, total_logabsdet

class SimpleMultiLevelImageTransform(MultiLevelImageTransform):
    def __init__(self, in_channels, height, width, kernel_size=3, hidden_channels=64, n_flows=3, n_scales=3):
        super().__init__(in_channels, height, width, [self.create_transform(in_channels * (2 ** i) * 2, height // (2 ** i), width // (2 ** i), kernel_size, hidden_channels, n_flows) for i in range(1, n_scales+1)])

    def create_transform(self, in_channels, height, width, kernel_size, hidden_channels, n_flows):
        transforms_list = []
        # stack n_flows coupling layers with ActNorm in between, using parity to alternate which pixels are transformed
        for i in range(n_flows):
            transforms_list.append(ActNorm(features=in_channels)) # OneByOneConvolution
            transforms_list.append(OneByOneConvolution(in_channels))
            transforms_list.append(Linear2DParityConvImageTransform(in_channels, height, width, kernel_size, parity=i))
            transforms_list.append(Linear2DParityConvImageTransform(in_channels, height, width, kernel_size, parity=i+1))
            # TODO replace this by a residual network based transform
            # transforms_list.append(SimpleTanh2DParityConvImageTransform(in_channels, height, width, hidden_channels=hidden_channels, kernel_size=kernel_size, parity=i))
            transforms_list.append(
                ResidualTanh2DParityConvImageTransform(
                    in_channels,
                    height,
                    width,
                    hidden_channels=hidden_channels,
                    kernel_size=kernel_size,
                    parity=i,
                    n_blocks=3,
                    residual_scale=0.1,
                )
            )
        transforms_list.append(ActNorm(features=in_channels))
        transforms_list.append(OneByOneConvolution(in_channels))
        transforms_list.append(Linear2DParityConvImageTransform(in_channels, height, width, kernel_size, parity=n_flows))
        transforms_list.append(Linear2DParityConvImageTransform(in_channels, height, width, kernel_size, parity=n_flows+1))
        transform = ImageCompositeTransform(in_channels, height, width, transforms_list)
        return transform

In [ ]:
K = 3
L = 128
n_flows = 6  # change to 6 for the depth experiment
n_scales = 3
learning_rate = 1e-4
if n_scales < 1 or H % 2**n_scales or W % 2**n_scales:
    raise ValueError("Image dimensions must be divisible by 2**n_scales")
model = Flow(
    transform=SimpleMultiLevelImageTransform(C, H, W, kernel_size=K,
        hidden_channels=L, n_flows=n_flows, n_scales=n_scales),
    distribution=StandardNormal(shape=[C, H, W]),
).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Initialize ActNorm with real training images and check the bijection.
model.train()
with torch.no_grad():
    probe = train_images[:min(4, len(train_images))].to(device)
    latent, ld = model._transform(probe)
    recovered, ild = model._transform.inverse(latent)
    assert latent.shape == probe.shape and ld.shape == (len(probe),)
    assert torch.isfinite(latent).all() and torch.isfinite(ld).all()
    torch.testing.assert_close(recovered, probe, atol=1e-4, rtol=1e-4)
    torch.testing.assert_close(ld + ild, torch.zeros_like(ld), atol=0.1, rtol=0)
    assert torch.isfinite(model.log_prob(probe)).all()
    print("Inverse max error:", (recovered-probe).abs().max().item())
    print("Logdet cancellation max error:", (ld+ild).abs().max().item())


In [ ]:
# Generate samples
@torch.no_grad()
def generate_samples(model, n_samples):
    samples = model.sample(n_samples).cpu()
    return samples

## Checkpoint geometry diagnostics

Each checkpointed epoch writes `geometry/epoch_XXXX/barycentres.png` and ten
`interpolation_to_val_XX.png` files inside the checkpoint directory. The two
barycentres use the entire fixed `val_images` batch, as in the later barycentre
analysis. Targets are `val_images[0:10]`, exactly the ten endpoints used by the
five later pairwise comparisons. Each interpolation figure has Euclidean above
and Riemannian below, both from the current Riemannian barycentre to the same target.

For the standard pullback metric used below, with image-to-latent map f,
the Riemannian barycentre is f^{-1}(mean_i f(x_i)) and its geodesic to x is
f^{-1}((1-t) mean_i f(x_i) + t f(x)). These are computed directly through the
flow, with no iterative barycentre/geodesic solve. This diagnostic does not
use the later isometrized geometry. Semantic improvement is something to
inspect, not a guarantee of the likelihood objective.

Inference is chunked, uses eval/no_grad, restores module modes, and preserves
the training RNG state. Only display images are clipped; geometry calculations
are not. `metadata.json` records checkpoint aliases, reference indices, inverse
roundoff, curve differences, and clipping fractions. Endpoint pixels are aligned
for display after recording inverse roundoff. Diagnostic failures stop execution
rather than silently skipping figures; model checkpoints have already been saved.
All checkpoint aliases written in an epoch share one set of figures.

Run the notebook from the top. If resuming, diagnostics start at the next trained
epoch; old epochs are not reconstructed automatically. To render any loaded
checkpoint manually, call `save_geometry_snapshot(phi, checkpoint['epoch'] + 1,
checkpoint_dir, ['best_model.pth'])` after loading it (change the name if needed).


In [ ]:
# Fixed geometry diagnostics, evaluated once per checkpointed epoch.
# The full validation batch defines the barycentres, matching the analysis below.
geometry_reference_images = val_images.detach().cpu().clone()
geometry_target_indices = tuple(range(10))
geometry_num_steps = 10
geometry_eval_batch_size = 2  # Reduce to 1 if diagnostic inference runs out of memory.
geometry_png_dpi = 140

@torch.no_grad()
def save_geometry_snapshot(phi, epoch_number, save_dir, checkpoint_names):
    import json

    if len(geometry_reference_images) <= max(geometry_target_indices):
        raise ValueError("Geometry diagnostics require the first ten validation images")
    if geometry_num_steps < 2 or geometry_eval_batch_size < 1:
        raise ValueError("Use >=2 interpolation steps and a positive inference batch size")
    parameter = next(phi.parameters())
    dev, dtype = parameter.device, parameter.dtype
    output_dir = Path(save_dir) / "geometry" / f"epoch_{epoch_number:04d}"
    output_dir.mkdir(parents=True, exist_ok=True)
    modes = [(module, module.training) for module in phi.modules()]
    cuda_devices = [dev.index if dev.index is not None else torch.cuda.current_device()] if dev.type == "cuda" else []

    def check_finite(x, name):
        if not torch.isfinite(x).all():
            raise FloatingPointError(f"Nonfinite {name} in geometry snapshot {epoch_number}")
        return x

    def apply_transform(images, inverse=False):
        # Keep the complete reference batch and plotting tensors on CPU.
        operation = phi.nflow._transform.inverse if inverse else phi.nflow._transform
        pieces = []
        for chunk in images.split(geometry_eval_batch_size):
            values, _ = operation(chunk.to(device=dev, dtype=dtype))
            pieces.append(check_finite(values, "transform output").detach().cpu())
        return torch.cat(pieces, dim=0)

    def save_figure(fig, name):
        try:
            fig.savefig(output_dir / name, dpi=geometry_png_dpi, bbox_inches="tight")
        finally:
            plt.close(fig)

    def clipped_fraction(images):
        mean = images.new_tensor(image_mean).view(1, C, 1, 1)
        std = images.new_tensor(image_std).view(1, C, 1, 1)
        pixels = images * std + mean
        return ((pixels < 0) | (pixels > 1)).float().mean().item()

    try:
        phi.eval()
        # Diagnostics must not advance the training RNG stream.
        with torch.random.fork_rng(devices=cuda_devices):
            references = geometry_reference_images.to(dtype=dtype)
            z = apply_transform(references)
            euclidean_barycentre = references.mean(dim=0, keepdim=True)
            z_barycentre = z.mean(dim=0, keepdim=True)
            riemannian_barycentre = apply_transform(z_barycentre, inverse=True)
            recovered_targets = apply_transform(z[list(geometry_target_indices)], inverse=True)
            targets = references[list(geometry_target_indices)]
            barycentre_latent_error = (apply_transform(riemannian_barycentre) - z_barycentre).abs().max().item()
            inverse_target_error = (recovered_targets - targets).abs().max().item()

            fig, axes = plt.subplots(1, 2, figsize=(7, 4))
            for ax, image, title in zip(axes,
                    [euclidean_barycentre[0], riemannian_barycentre[0]],
                    ["Euclidean barycentre", "Riemannian barycentre"]):
                ax.imshow(display_image(image))
                ax.set_title(title, fontsize=12)
                ax.axis("off")
            fig.suptitle(f"Epoch {epoch_number} | same {len(references)} validation images", fontsize=12)
            fig.tight_layout()
            save_figure(fig, "barycentres.png")

            times = torch.linspace(0, 1, geometry_num_steps, dtype=dtype)
            weights = times.view(-1, 1, 1, 1)
            target_metrics = []
            for target_index in geometry_target_indices:
                target = references[target_index:target_index + 1]
                linear = (1 - weights) * riemannian_barycentre + weights * target
                latent_path = (1 - weights) * z_barycentre + weights * z[target_index:target_index + 1]
                geodesic = apply_transform(latent_path, inverse=True)
                endpoint_error = max(
                    (geodesic[0:1] - riemannian_barycentre).abs().max().item(),
                    (geodesic[-1:] - target).abs().max().item())
                # Show exactly identical endpoints; record inverse roundoff before replacement.
                geodesic[0], geodesic[-1] = riemannian_barycentre[0], target[0]
                linear[0], linear[-1] = riemannian_barycentre[0], target[0]
                fig, axes = plt.subplots(2, geometry_num_steps,
                    figsize=(1.65 * geometry_num_steps, 4.5), squeeze=False)
                for row, images in enumerate([linear, geodesic]):
                    for j, ax in enumerate(axes[row]):
                        ax.imshow(display_image(images[j]))
                        ax.set_xticks([])
                        ax.set_yticks([])
                        for spine in ax.spines.values():
                            spine.set_visible(False)
                        if row == 0:
                            ax.set_title(f"t={times[j].item():.2f}", fontsize=9)
                    axes[row, 0].set_ylabel(["Euclidean", "Riemannian"][row], fontsize=11)
                fig.suptitle(f"Epoch {epoch_number} | Riemannian barycentre to validation image {target_index}\n"
                             "Same start, same target, same parameter values", fontsize=12)
                fig.tight_layout(rect=(0, 0, 1, 0.88))
                save_figure(fig, f"interpolation_to_val_{target_index:02d}.png")
                target_metrics.append({
                    "validation_batch_index": target_index,
                    "endpoint_max_abs_error_before_display_alignment": endpoint_error,
                    "curve_difference_rms_normalized_pixels": (geodesic - linear).square().mean().sqrt().item(),
                    "euclidean_display_clipped_fraction": clipped_fraction(linear),
                    "riemannian_display_clipped_fraction": clipped_fraction(geodesic),
                })

            metadata = {
                "epoch": epoch_number, "checkpoint_files": list(checkpoint_names),
                "data_id": data_id, "reference_count": len(references),
                "reference_dataset_indices": [int(i) for i in data_val.indices[:len(references)]],
                "target_validation_batch_indices": list(geometry_target_indices),
                "interpolation_times": times.tolist(),
                "barycentre_metric": "Euclidean metric pulled back by phi.nflow._transform",
                "barycentre_latent_max_abs_error": barycentre_latent_error,
                "target_inverse_max_abs_error": inverse_target_error,
                "barycentre_difference_rms_normalized_pixels": (riemannian_barycentre - euclidean_barycentre).square().mean().sqrt().item(),
                "euclidean_barycentre_display_clipped_fraction": clipped_fraction(euclidean_barycentre),
                "riemannian_barycentre_display_clipped_fraction": clipped_fraction(riemannian_barycentre),
                "targets": target_metrics,
            }
            (output_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))
        print(f"Geometry snapshot: {output_dir} (11 PNGs)")
    finally:
        # Restore even mixed per-module modes, including on diagnostic failure.
        for module, training in modes:
            module.training = training


In [ ]:
@torch.no_grad()
def evaluate_nll(phi, loader):
    phi.eval()
    total, count = 0.0, 0
    for x, _ in loader:
        nll = -phi.nflow.log_prob(x.to(device))
        if not torch.isfinite(nll).all():
            raise FloatingPointError("Nonfinite held-out NLL")
        total += nll.sum().item()
        count += len(x)
    return total / count

# Same likelihood objective and training-loss checkpoint selection as the source.
def train(phi, train_loader, optimizer, n_epochs, save_dir='results', start_epoch=0,
          best_loss=float('inf'), geometry_callback=None):
    os.makedirs(save_dir, exist_ok=True)
    for epoch in range(start_epoch, n_epochs):
        phi.train()
        total_loss, count = 0.0, 0
        for x, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}"):
            x = x.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = -phi.nflow.log_prob(x).mean()
            if not torch.isfinite(loss):
                raise FloatingPointError('Nonfinite training loss')
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(x)
            count += len(x)
        avg_epoch_loss = total_loss / count
        validation_nll = evaluate_nll(phi, validation_loader)
        log_path = os.path.join(save_dir, "loss_history.csv")
        exists = os.path.isfile(log_path)
        with open(log_path, "a", newline="") as f:
            writer = csv.writer(f)
            if not exists:
                writer.writerow(["epoch", "train_nll", "validation_nll"])
            writer.writerow([epoch+1, avg_epoch_loss, validation_nll])
        improved = avg_epoch_loss < best_loss
        best_loss = min(best_loss, avg_epoch_loss)
        checkpoint = {
            'epoch': epoch, 'model_state_dict': phi.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_epoch_loss, 'best_loss': best_loss,
            'validation_nll': validation_nll,
            'torch_rng_state': torch.get_rng_state(),
        }
        if torch.cuda.is_available():
            checkpoint['cuda_rng_state_all'] = torch.cuda.get_rng_state_all()
        saved_checkpoint_names = ['last_model.pth']
        torch.save(checkpoint, os.path.join(save_dir, 'last_model.pth'))
        if (epoch + 1) % 5 == 0:
            torch.save(checkpoint, os.path.join(save_dir, f'epoch_{epoch+1}_model.pth'))
            saved_checkpoint_names.append(f'epoch_{epoch+1}_model.pth')
        if improved:
            torch.save(checkpoint, os.path.join(save_dir, 'best_model.pth'))
            saved_checkpoint_names.append('best_model.pth')
        if geometry_callback is not None:
            geometry_callback(phi, epoch + 1, save_dir, saved_checkpoint_names)
        print(f"Epoch {epoch+1}: NLL/image={avg_epoch_loss:.4f}; validation={validation_nll:.6f}; best train={best_loss:.6f}")


In [ ]:
n_epochs = 50
phi = NFlowImageDiffeomorphism(C, H, W, model)
checkpoint_dir = os.path.join(results_folder,
    f"hidden={L}_kernel={K}_flows={n_flows}_scales={n_scales}_lr={learning_rate}_batch={batch_size}_data={data_id}")
os.makedirs(checkpoint_dir, exist_ok=True)
manifest_path = Path(checkpoint_dir) / "data_manifest.json"
if manifest_path.exists() and manifest_path.read_text() != manifest_text:
    raise RuntimeError("Checkpoint data manifest mismatch")
manifest_path.write_text(manifest_text)
best_model_path = os.path.join(checkpoint_dir, 'best_model.pth')
if train_model:
    start_epoch, best_loss = 0, float('inf')
    last_path = os.path.join(checkpoint_dir, 'last_model.pth')
    if resume_training and os.path.isfile(last_path):
        checkpoint = torch.load(last_path, map_location=device, weights_only=True)
        phi.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_loss = checkpoint['best_loss']
        torch.set_rng_state(checkpoint['torch_rng_state'].cpu())
        if torch.cuda.is_available() and 'cuda_rng_state_all' in checkpoint:
            torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint['cuda_rng_state_all']])
    train(phi, train_loader, optimizer, n_epochs, checkpoint_dir, start_epoch, best_loss,
          geometry_callback=save_geometry_snapshot)
if not os.path.isfile(best_model_path):
    raise FileNotFoundError('No best checkpoint. Set train_model=True to train first.')
checkpoint = torch.load(best_model_path, map_location=device, weights_only=True)
phi.load_state_dict(checkpoint['model_state_dict'])
phi.eval()


In [ ]:
# Visualize generated samples
samples = generate_samples(phi.nflow, 6).reshape(-1, C, H, W)

# Plot the first 6 images
fig, axes = plt.subplots(1, 6, figsize=(12, 8))
axes = axes.flatten()

for i in range(6):
    axes[i].imshow(display_image(samples[i].squeeze()))
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# special points
num_points = 10
num_geodesics = 5

x0 = val_images[0:2*num_geodesics:2].to(device)
x1 = val_images[1:2*num_geodesics+1:2].to(device)

### Naive Riemannian data analysis ###

In [ ]:
# Construct pullback manifold
manifold = StandardPullbackImageEuclidean(phi)

In [ ]:
# Compute geodesics
t = torch.linspace(0., 1., num_points).to(device)

geodesics = torch.zeros(num_geodesics, num_points, C, H, W)

# Generate geodesics
for i in range(num_geodesics):
    geodesics[i] = manifold.geodesic(x0[i][None,None], x1[i][None,None], t).detach().cpu()[0,0,0]

In [ ]:
# Plot each image in the grid
for i in range(num_geodesics):
    fig, axes = plt.subplots(1,num_points, figsize=(5*num_points, 5))
    for j in range(num_points):
        axes[j].imshow(display_image(geodesics[i, j].squeeze()))
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(results_folder,f"geodesic-{i}.eps"))
    plt.show()

In [ ]:
# Compute barycentre
bary = manifold.barycentre(val_images.to(device)).detach().cpu()

In [ ]:
# Plot barycentres
plt.imshow(display_image(bary.squeeze()))
plt.axis('off')
plt.show()

In [ ]:
# Construct curvature corrected l2-pga solver
l2_pga_solver = l2TangentSpacePCAImageSolver(val_images.to(device), manifold, bary.to(device))

In [ ]:
# Compute low rank approximations
Xi_bary, exp_bary_Xi, _ = l2_pga_solver.solve(pga_rank)

In [ ]:
# Plot the first 6 images
fig, axes = plt.subplots(1, num_points, figsize=(5*num_points, 5))
axes = axes.flatten()
for i in range(num_points):
    axes[i].imshow(display_image(val_images[i].squeeze()))
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(results_folder,"val-l2-pga.eps"))
plt.show()

fig, axes = plt.subplots(1, num_points, figsize=(5*num_points, 5))
axes = axes.flatten()
for i in range(num_points):
    axes[i].imshow(display_image(exp_bary_Xi.squeeze().detach().cpu().numpy()[i]))
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(results_folder,"l2-pga.eps"))
plt.show()

### Isometrized Riemannian data analysis ###

In [ ]:
# Construct isometrized pullback manifold
num_intervals = 5  # You can adjust this value as needed
iso_manifold = l2IsometrizedImageEuclidean(manifold, num_intervals=num_intervals)

In [ ]:
# Compute geodesics
t = torch.linspace(0., 1., num_points).to(device)

iso_geodesics = torch.zeros(num_geodesics, num_points, C, H, W)

# Generate geodesics
for i in range(num_geodesics):
    iso_geodesics[i] = iso_manifold.geodesic(x0[i][None,None], x1[i][None,None], t).detach().cpu()[0,0,0]

In [ ]:
# Plot each image in the grid
for i in range(num_geodesics):
    fig, axes = plt.subplots(1, num_points, figsize=(5*num_points, 5))
    for j in range(num_points):
        axes[j].imshow(display_image(iso_geodesics[i, j].squeeze()))
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(results_folder,f"iso-geodesic-{i}-M{num_intervals}.eps"))
    plt.show()

In [ ]:
# Construct curvature corrected l2-pga solver
iso_l2_pga_solver = l2TangentSpacePCAImageSolver(val_images.to(device), iso_manifold, bary.to(device))

In [ ]:
# Compute low rank approximations
iso_Xi_bary, iso_exp_bary_Xi, _ = iso_l2_pga_solver.solve(pga_rank)

In [ ]:
# Plot the first 6 images
fig, axes = plt.subplots(1, num_points, figsize=(5*num_points, 5))
axes = axes.flatten()
for i in range(num_points):
    axes[i].imshow(display_image(val_images[i].squeeze()))
    axes[i].axis('off')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, num_points, figsize=(5*num_points, 5))
axes = axes.flatten()
for i in range(num_points):
    axes[i].imshow(display_image(iso_exp_bary_Xi.squeeze().detach().cpu().numpy()[i]))
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(results_folder,f"iso-l2-pga-M{num_intervals}.eps"))
plt.show()

### l2 data analysis ###

In [ ]:
# Construct manifold
l2_manifold = StandardImageEuclidean(C, H, W)

In [ ]:
# Compute geodesics
t = torch.linspace(0., 1., num_points).to(device)

l2_geodesics = torch.zeros(num_geodesics, num_points, C, H, W)

# Generate geodesics
for i in range(num_geodesics):
    l2_geodesics[i] = l2_manifold.geodesic(x0[i][None,None], x1[i][None,None], t).detach().cpu()[0,0,0]

In [ ]:
# Plot each image in the grid
for i in range(num_geodesics):
    fig, axes = plt.subplots(1, num_points, figsize=(5*num_points, 5))
    for j in range(num_points):
        axes[j].imshow(display_image(l2_geodesics[i, j].squeeze()))
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(results_folder,f"l2-geodesic-{i}.eps"))
    plt.show()

In [ ]:
# Construct curvature corrected l2-pga solver
l2_pca_solver = l2TangentSpacePCAImageSolver(val_images.to(device), l2_manifold, bary.to(device))

In [ ]:
# Compute low rank approximations
l2_Xi_bary, l2_exp_bary_Xi, _ = l2_pca_solver.solve(pga_rank)

In [ ]:
# Plot the first 6 images
fig, axes = plt.subplots(1, num_points, figsize=(5*num_points, 5))
axes = axes.flatten()
for i in range(num_points):
    axes[i].imshow(display_image(val_images[i].squeeze()))
    axes[i].axis('off')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, num_points, figsize=(5*num_points, 5))
axes = axes.flatten()
for i in range(num_points):
    axes[i].imshow(display_image(l2_exp_bary_Xi.squeeze().detach().cpu().numpy()[i]))
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(results_folder,"l2-pca.eps"))
plt.show()

### Comparing naive and isometrized ###

In [ ]:
# Compute geodesic relative rmse from the barycentre
squared_geo_errors = torch.zeros(len(val_images))
dists = ((val_images - bary)**2).sum([1,2,3])
for i, x_i in enumerate(val_images):
    geodesic_i = manifold.geodesic(bary.to(device)[None,None], x_i.to(device)[None,None], t).detach().cpu()[0,0,0]
    iso_geodesic_i = iso_manifold.geodesic(bary.to(device)[None,None], x_i.to(device)[None,None], t).detach().cpu()[0,0,0]
    squared_geo_errors[i] = ((geodesic_i - iso_geodesic_i)**2).mean(0).sum()

geo_rel_rmse = (squared_geo_errors.mean() / dists.mean()).sqrt()
print(geo_rel_rmse)

In [ ]:
# Make scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(dists.sqrt(), squared_geo_errors.sqrt())
plt.xlim([0.,dists.sqrt().max()*1.1])
# Use automatic y limits for RGB data.
plt.xlabel('distance from barycentre')
plt.ylabel('geodesic error')
plt.savefig(os.path.join(results_folder,f"geo-error-M{num_intervals}.eps"))
plt.show()

In [ ]:
# Compute low rank relative rmse from the barycentre
squared_low_rank_errors = ((val_images - exp_bary_Xi.cpu())**2).sum([1,2,3])
low_rank_rel_rmse = (squared_low_rank_errors.mean() / dists.mean()).sqrt()

squared_low_rank_iso_errors = ((val_images - iso_exp_bary_Xi.cpu())**2).sum([1,2,3])
iso_low_rank_rel_rmse = (squared_low_rank_iso_errors.mean() / dists.mean()).sqrt()

squared_low_rank_l2_errors = ((val_images - l2_exp_bary_Xi.cpu())**2).sum([1,2,3])
l2_low_rank_rel_rmse = (squared_low_rank_l2_errors.mean() / dists.mean()).sqrt()

print(low_rank_rel_rmse)
print(iso_low_rank_rel_rmse)
print(l2_low_rank_rel_rmse)

In [ ]:
# Make scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(dists.sqrt(), squared_low_rank_errors.sqrt() - squared_low_rank_iso_errors.sqrt())
plt.xlim([0.,dists.sqrt().max()*1.1])
# Use automatic y limits for RGB data.
plt.xlabel('distance from barycentre')
plt.ylabel('low-rank error')
plt.savefig(os.path.join(results_folder,f"low-rank-error-M{num_intervals}.eps"))
plt.show()